## Bibliotecas

In [6]:
import numpy as np
import torch
import torch.nn.functional as F
import torch.nn as nn

## Breve História sobre as CNNs e modelos mais utilizados


Desde o surgimento das redes neurais convolucionais muitos modelos foram propostos. Segue aqui um pequeno texto da 'ordem evolutiva' desses modelos:

Primeira rede neural convolucional veio em 1989 por Yann LeCun, em 1998 houve um grande avanço nos modelos surgindo a LeNet contudo até 2012 as redes neurais ficaram em um limbo devido aluns fatores, entre eles estão:
- Não existiam datasets grandes o suficiente para demonstrar as capacidades da CNN
- Poder computacional da época era limitado
- Para datasets simples e pequenos algums modelos como SVM performavam melhor do que CNN

Contudo em 2012 várias das limitações foram resolvidas, e surge um nova arquitetura de CNN chamada de AlexNet craido por Alex Krizhevsky

Estrutura AlexNet:
- 2 blocos de (Convolução - Max Pooling)
- 3 camadas de Convolução
- Max Pooling
- 4 camadas densas com saída softamx
- Ativação do Modelo Foi ReLu

Com o grande impacto das CNNs em 2014 3 modelos importantes foram desenvolvidos:
- GoogleNet/Inception v1: Introduzio módulos Inception, que aplicam diferentes tamanhos de filtros em paralelo
- VGG: Mudança foi kernels menores e maiores profundiades
- Inception v3: Uso de Batch Normalization e Kernels não quadráticos (Como 1x7, 7x1, 1x3, 3x1)

Em 2015 surge o ResNet que trás uma avanço significativo com os skip connections. Essa nova ideia surgiu para resolver um problema em CNNs muito profundas, na qual o gradiente decaia ao ponto de ser muito pequeno nas primeiras camadas, sendo assim elas eram conectadas com camadas mais adiantes, fazendo concateção dos valores:

![image.png](https://www.researchgate.net/profile/Ajita-Rattani/publication/354665713/figure/fig3/AS:1069132669788161@1631912500322/The-architecture-of-the-ResNet-50-based-deep-learning-model-with-skip-connection.ppm)

Uma extensão das ResNet vieram logo em seguida em 2016, na qual cada camada é conectada a todas as camdas anteiores.

![image.png](https://www.researchgate.net/profile/Hammam-Alshazly/publication/353556852/figure/fig3/AS:1050959069339648@1627579576058/A-schematic-diagram-of-a-3-layer-dense-block-used-in-the-DenseNet-architecture-Full-size.png)

Em 2017 tivemos as arquiteturas hibridas como as ResNeXt, em 2019 e 2020 tivemos as EfficientNet que teve a inovação em um método eficiente de escalonamento da rede chamado de 'Compound Scaling'. Antes a maioria das arquitetura aumentava profundiade, largura ou reoslução da imagem separadamente. EfficientNet escalona os três ao mesmo tempo de maneira balanceada, melhorando eficiência.

Contudo após 2020 o domínio das CNN diminuiram por causa do surgimento de uma nova arquitetura ainda mais potente as Vision Transformers (ViTs)

Caso queiram utilizar esses modelos no pytorch eles já estão prontos:
https://pytorch.org/vision/stable/models.html

## Código Suplementar - Bloco Inception e Skip Connection

Abaixo está o código de como fazer um bloco incpetion e um skip connection:

Bloco Inception:

![image.png](https://www.researchgate.net/profile/Anabia-Sohail-2/publication/330511306/figure/fig4/AS:717351204958208@1548041263007/Basic-architecture-of-inception-block.ppm)

In [2]:
class InceptionModule(nn.Module):
  # Define uma nova classe chamada 'InceptionModule' que herda de 'nn.Module'.
  # Este módulo implementa a arquitetura Inception, que consiste em diferentes caminhos de convolução paralelos.
  def __init__(self, input, n_channels1x1, n_channels3x3first, n_channels3x3, n_channels5x5first, n_channels5x5, pooling):
    # '__init__' é o construtor da classe. Ele é chamado quando um objeto 'InceptionModule' é criado.
    # 'self' é uma referência à instância da classe que está sendo criada.
    # 'input' é o número de canais de entrada para este módulo Inception.
    # 'n_channels1x1' é o número de filtros na camada de convolução 1x1.
    # 'n_channels3x3first' é o número de filtros na primeira camada de convolução 1x1 que precede a convolução 3x3.
    # 'n_channels3x3' é o número de filtros na camada de convolução 3x3.
    # 'n_channels5x5first' é o número de filtros na primeira camada de convolução 1x1 que precede a convolução 5x5.
    # 'n_channels5x5' é o número de filtros na camada de convolução 5x5.
    # 'pooling' é o número de filtros na camada de convolução 1x1 que segue a camada de Max Pooling.
    super(InceptionModule, self).__init__()
    # 'super(InceptionModule, self).__init__()' chama o construtor da classe pai ('nn.Module') para inicialização adequada.

    # Convolução 1x1
    self.bloco1 = nn.Sequential(
        # 'self.bloco1' define o primeiro bloco do módulo Inception.
        # Este bloco consiste em uma única camada de convolução 1x1.
        nn.Conv2d(input, n_channels1x1, kernel_size=1),
        # 'nn.Conv2d' aplica uma convolução 2D.
        # 'input': número de canais de entrada para esta camada.
        # 'n_channels1x1': número de filtros (canais de saída) desta camada de convolução 1x1.
        # 'kernel_size=1': especifica um filtro de tamanho 1x1, que realiza uma combinação linear dos canais de entrada sem alterar as dimensões espaciais.
        nn.BatchNorm2d(n_channels1x1),
        # 'nn.BatchNorm2d' aplica Batch Normalization sobre os 'n_channels1x1' canais de saída da convolução.
        # Isso ajuda a estabilizar o treinamento e acelerar a convergência.
        nn.ReLU(True)
        # 'nn.ReLU(True)' aplica a função de ativação ReLU (Rectified Linear Unit) de forma in-place (modificando o tensor diretamente para economizar memória).
        # A ReLU introduz não-linearidade na rede.
    )

    # Convolução 1x1 -> Convolução 3x3
    self.bloco2 = nn.Sequential(
        # 'self.bloco2' define o segundo bloco do módulo Inception.
        # Este bloco consiste em uma convolução 1x1 seguida por uma convolução 3x3.
        nn.Conv2d(input, n_channels3x3first, kernel_size=1),
        # Primeira camada de convolução 1x1 para reduzir a dimensionalidade do número de canais antes da convolução 3x3, tornando-a mais eficiente computacionalmente.
        # 'input': número de canais de entrada.
        # 'n_channels3x3first': número de filtros de saída desta camada 1x1.
        # 'kernel_size=1': filtro 1x1.
        nn.BatchNorm2d(n_channels3x3first),
        # Batch Normalization para a saída da primeira convolução.
        nn.ReLU(True),
        # Ativação ReLU após a primeira Batch Normalization.
        nn.Conv2d(n_channels3x3first, n_channels3x3, kernel_size=3, padding=1),
        # Segunda camada de convolução 3x3.
        # 'n_channels3x3first': número de canais de entrada (saída da camada anterior).
        # 'n_channels3x3': número de filtros de saída desta camada 3x3.
        # 'kernel_size=3': filtro de tamanho 3x3, que considera uma vizinhança maior na imagem.
        # 'padding=1': adiciona um padding de 1 pixel ao redor da entrada, garantindo que a saída tenha as mesmas dimensões espaciais da entrada (após a convolução).
        nn.BatchNorm2d(n_channels3x3),
        # Batch Normalization para a saída da segunda convolução.
        nn.ReLU(True)
        # Ativação ReLU após a segunda Batch Normalization.
    )

    # Convolução 1x1 -> Convolução 5x5
    self.bloco3 = nn.Sequential(
        # 'self.bloco3' define o terceiro bloco do módulo Inception.
        # Este bloco consiste em uma convolução 1x1 seguida por uma convolução 5x5.
        nn.Conv2d(input, n_channels5x5first, kernel_size=1),
        # Primeira camada de convolução 1x1 para reduzir a dimensionalidade antes da convolução 5x5.
        # 'input': número de canais de entrada.
        # 'n_channels5x5first': número de filtros de saída desta camada 1x1.
        # 'kernel_size=1': filtro 1x1.
        nn.BatchNorm2d(n_channels5x5first),
        # Batch Normalization para a saída da primeira convolução.
        nn.ReLU(True),
        # Ativação ReLU após a primeira Batch Normalization.
        nn.Conv2d(n_channels5x5first, n_channels5x5, kernel_size=5, padding=2),
        # Segunda camada de convolução 5x5.
        # 'n_channels5x5first': número de canais de entrada.
        # 'n_channels5x5': número de filtros de saída desta camada 5x5.
        # 'kernel_size=5': filtro de tamanho 5x5, que considera uma vizinhança ainda maior.
        # 'padding=2': adiciona um padding de 2 pixels ao redor da entrada, mantendo as dimensões espaciais da saída iguais às da entrada.
        nn.BatchNorm2d(n_channels5x5),
        # Batch Normalization para a saída da segunda convolução.
        nn.ReLU(True)
        # Ativação ReLU após a segunda Batch Normalization.
    )

    # Max Pooling 3x3 -> Convolução 1x1
    self.bloco4 = nn.Sequential(
            # 'self.bloco4' define o quarto bloco do módulo Inception.
            # Este bloco consiste em uma camada de Max Pooling seguida por uma convolução 1x1.
            nn.MaxPool2d(3, stride=1, padding=1),
            # 'nn.MaxPool2d': aplica Max Pooling 2D.
            # 'kernel_size=3': janela de pooling de tamanho 3x3.
            # 'stride=1': o passo da janela de pooling é de 1 pixel.
            # 'padding=1': adiciona um padding de 1 pixel à entrada antes do pooling. Isso garante que a saída do pooling tenha as mesmas dimensões espaciais da entrada.
            nn.Conv2d(input, pooling, kernel_size=1),
            # Camada de convolução 1x1 para reduzir o número de canais após o pooling.
            # 'input': número de canais de entrada (o mesmo da entrada do módulo).
            # 'pooling': número de filtros de saída desta camada 1x1 (definido pelo argumento 'pooling' do construtor).
            # 'kernel_size=1': filtro 1x1.
            nn.BatchNorm2d(pooling),
            # Batch Normalization para a saída da convolução 1x1.
            nn.ReLU(True)
            # Ativação ReLU após a Batch Normalization.
        )

  def forward(self, ip):
    # 'forward' define a passagem forward dos dados através do módulo Inception.
    # 'self' é a referência à instância da classe.
    # 'ip' é o tensor de entrada para o módulo.
    op1 = self.bloco1(ip)
    # A entrada 'ip' passa pelo primeiro bloco (convolução 1x1).
    op2 = self.bloco2(ip)
    # A entrada 'ip' passa pelo segundo bloco (convolução 1x1 -> convolução 3x3).
    op3 = self.bloco3(ip)
    # A entrada 'ip' passa pelo terceiro bloco (convolução 1x1 -> convolução 5x5).
    op4 = self.bloco4(ip)
    # A entrada 'ip' passa pelo quarto bloco (Max Pooling -> convolução 1x1).
    return torch.cat([op1,op2,op3,op4], 1)
    # 'torch.cat([op1, op2, op3, op4], 1)' concatena as saídas dos quatro blocos ao longo da dimensão dos canais (dim=1).
    # Isso significa que os mapas de características produzidos por cada caminho de convolução são combinados em um único tensor de saída.

def testInceptionBlock():
  # 'testInceptionBlock' é uma função para testar o funcionamento do módulo Inception.
  x = torch.randn((32,192,28,28))
  # 'torch.randn((32, 192, 28, 28))' cria um tensor randômico que simula um lote de 32 imagens, cada uma com 192 canais, altura de 28 pixels e largura de 28 pixels.
  model = InceptionModule(192,64,96,128,16,32,32)
  # 'InceptionModule(192, 64, 96, 128, 16, 32, 32)' cria uma instância do módulo Inception com os seguintes parâmetros:
  # input=192 (o número de canais de entrada corresponde ao número de canais do tensor 'x').
  # n_channels1x1=64 (64 filtros na primeira convolução 1x1).
  # n_channels3x3first=96 (96 filtros na primeira convolução 1x1 do segundo bloco).
  # n_channels3x3=128 (128 filtros na convolução 3x3 do segundo bloco).
  # n_channels5x5first=16 (16 filtros na primeira convolução 1x1 do terceiro bloco).
  # n_channels5x5=32 (32 filtros na convolução 5x5 do terceiro bloco).
  # pooling=32 (32 filtros na convolução 1x1 após o Max Pooling no quarto bloco).
  print(model(x).shape)
  # 'model(x)' passa o tensor de entrada 'x' através do módulo Inception.
  # '.shape' imprime as dimensões do tensor de saída produzido pelo módulo. Isso nos ajuda a verificar se as dimensões estão corretas após a passagem forward.
  return model
  # Retorna a instância do módulo Inception criado para o teste.

In [3]:
model = testInceptionBlock()
# Inicializa o modelo

torch.Size([32, 256, 28, 28])


In [4]:
class Bloco_ResNet(nn.Module):
  # Define uma nova classe chamada 'Bloco_ResNet' que herda de 'nn.Module'.
  # Este bloco implementa a unidade básica residual da arquitetura ResNet.
  def __init__(self, input, out_channels, stride=1):
    # '__init__' é o construtor da classe. Ele é chamado quando um objeto 'Bloco_ResNet' é criado.
    # 'self' é uma referência à instância da classe que está sendo criada.
    # 'input' é o número de canais de entrada para este bloco residual.
    # 'out_channels' é o número de canais de saída que este bloco produzirá.
    # 'stride' é o passo da convolução na primeira camada convolucional. Um stride maior que 1 reduz as dimensões espaciais. O valor padrão é 1.
    super(Bloco_ResNet, self).__init__()
    # 'super(Bloco_ResNet, self).__init__()' chama o construtor da classe pai ('nn.Module') para inicialização adequada.

    self.conv1 = nn.Conv2d(in_channels=input, out_channels=out_channels,
                           kernel_size=3, stride=stride, padding=1, bias=False)
    # 'self.conv1' define a primeira camada de convolução 2D dentro do bloco residual.
    # 'nn.Conv2d': aplica uma operação de convolução 2D.
    # 'in_channels=input': o número de canais de entrada para esta camada.
    # 'out_channels=out_channels': o número de filtros (canais de saída) desta camada.
    # 'kernel_size=3': o tamanho do filtro de convolução é 3x3.
    # 'stride=stride': o passo do filtro durante a convolução. Se 'stride' for maior que 1, a dimensão espacial da saída será reduzida.
    # 'padding=1': adiciona um padding de 1 pixel ao redor da entrada, mantendo as dimensões espaciais da saída iguais às da entrada (se stride=1).
    # 'bias=False': especifica que esta camada de convolução não terá um termo de bias. O bias é geralmente redundante quando se usa Batch Normalization.

    self.batch_norm1 = nn.BatchNorm2d(out_channels)
    # 'self.batch_norm1' define a primeira camada de Batch Normalization, aplicada após a primeira convolução.
    # 'nn.BatchNorm2d': normaliza as saídas da camada convolucional anterior ('out_channels' canais).
    # A normalização em lote ajuda a estabilizar o treinamento.

    self.conv2 = nn.Conv2d(in_channels=out_channels, out_channels=out_channels,
                           kernel_size=3, stride=1, padding=1, bias=False)
    # 'self.conv2' define a segunda camada de convolução 2D dentro do bloco residual.
    # 'in_channels=out_channels': o número de canais de entrada é o mesmo do número de canais de saída da primeira convolução.
    # 'out_channels=out_channels': o número de filtros de saída desta camada é o mesmo da camada anterior.
    # 'kernel_size=3': tamanho do filtro 3x3.
    # 'stride=1': o passo é sempre 1 nesta segunda convolução nos blocos residuais básicos.
    # 'padding=1': adiciona padding para manter as dimensões espaciais.
    # 'bias=False': sem termo de bias.

    self.batch_norm2 = nn.BatchNorm2d(out_channels)
    # 'self.batch_norm2' define a segunda camada de Batch Normalization, aplicada após a segunda convolução.

    self.res_connnection = nn.Sequential()
    # 'self.res_connnection' define a conexão residual (shortcut connection).
    # Inicialmente, é um contêiner vazio ('nn.Sequential').

    if stride > 1 or input != out_channels:
      # Esta condição verifica se a dimensão espacial ou o número de canais precisa ser ajustado para que a saída do bloco possa ser somada à entrada (para a conexão residual).
      # Isso é necessário quando a primeira convolução tem um stride maior que 1 (reduzindo a dimensão espacial) ou quando o número de canais de entrada é diferente do número de canais de saída.
      self.res_connnection = nn.Sequential(
          nn.Conv2d(in_channels=input, out_channels=out_channels, kernel_size=1,
                    stride=stride, bias=False),
          # Se a condição for verdadeira, uma camada de convolução 1x1 é usada na conexão residual para corresponder ao número de canais de saída ('out_channels') e ajustar a dimensão espacial (se 'stride' > 1).
          # 'kernel_size=1': uma convolução 1x1 não altera as dimensões espaciais diretamente, mas pode alterar o número de canais.
          # 'stride=stride': usa o mesmo stride da primeira convolução para corresponder à redução espacial.
          # 'bias=False': sem termo de bias.
          nn.BatchNorm2d(out_channels)
          # Batch Normalization aplicada após a convolução 1x1 na conexão residual.
      )

  def forward(self, inp):
    # 'forward' define a passagem forward dos dados através do bloco residual.
    # 'self' é a referência à instância da classe.
    # 'inp' é o tensor de entrada para o bloco.

    op = F.relu(self.batch_norm1(self.conv1(inp)))
    # A entrada 'inp' passa pela primeira convolução ('self.conv1'), depois pela Batch Normalization ('self.batch_norm1'), e finalmente pela função de ativação ReLU ('F.relu').
    # 'F.relu()' aplica a função ReLU (max(0, x)).

    op = self.batch_norm2(self.conv2(op))
    # A saída da primeira ativação passa pela segunda convolução ('self.conv2') e depois pela segunda Batch Normalization ('self.batch_norm2').
    # Note que a ativação ReLU não é aplicada aqui imediatamente.

    op += self.res_connnection(inp)
    # A saída do segundo Batch Normalization é somada à saída da conexão residual ('self.res_connnection(inp)').
    # Se a dimensão espacial ou o número de canais foi alterado na conexão residual, a soma é feita entre tensores de dimensões correspondentes.
    # Esta é a parte fundamental do bloco residual: adicionar a entrada original (ou uma versão transformada dela) à saída do caminho principal. Isso ajuda a evitar o problema do desaparecimento de gradiente em redes profundas.

    op = F.relu(op)
    # Finalmente, a função de ativação ReLU é aplicada à soma.

    return op
    # Retorna a saída do bloco residual.

In [7]:
# Cria um tensor (estrutura de dados usada no PyTorch, similar a uma matriz) com valores aleatórios.
# Esse tensor simula uma imagem ou mapa de ativação com as seguintes dimensões:
# - 1: tamanho do batch (quantidade de amostras processadas de uma vez)
# - 64: quantidade de canais (como profundidade; ex: em imagens RGB são 3 canais, aqui temos 64)
# - 32: altura da imagem
# - 32: largura da imagem
x = torch.randn(1, 64, 32, 32)

# Aqui é criada uma instância do bloco residual (comumente usado na arquitetura ResNet).
# - O bloco espera 64 canais de entrada e gera 64 canais de saída.
# - Isso significa que a entrada e a saída terão o mesmo formato, ideal para "residual connections".
modelo = Bloco_ResNet(64, 64)

# O tensor x (imagem simulada) é passado pelo modelo.
# A saída é armazenada na variável `saida`.
saida = modelo(x)

# Exibe o formato (shape) da saída após passar pelo bloco ResNet.
# Isso ajuda a verificar se as dimensões permanecem corretas após o processamento.
print(saida.shape)

torch.Size([1, 64, 32, 32])


## Exercício

Abaixo vou realizar a implementação de uma das CNNs mais famosas a VGG16, em seguida vou colocar 2 arquiteturas que servem de exercício para práttica, uma mais simples (AlexNet) e outra mais difícil (GoogLenet)

### VGG 16

![image.png](https://raw.githubusercontent.com/blurred-machine/Data-Science/master/Deep%20Learning%20SOTA/img/network.png)

In [8]:
class VGG16(nn.Module):
    def __init__(self, num_classes=1000):
        super(VGG16, self).__init__()

        self.features = nn.Sequential(
            # Bloco 1: Size 224 -> 112
            nn.Conv2d(3, 64, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Bloco 2: Size 112 -> 56
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Bloco 3: Size 56 -> 28
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Bloco 4: Size 28 -> 14
            nn.Conv2d(256, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Bloco 5: Size 14 -> 7
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # Bloco 6: Camadas Fully Connected
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, 4096), nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(4096, 4096), nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(4096, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

def testVGG16():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = VGG16(num_classes=1000).to(device)
    input_tensor = torch.randn(1, 3, 224, 224).to(device)
    with torch.no_grad():
        output = model(input_tensor)

    print(f"Shape da entrada: {input_tensor.shape}")
    print(f"Shape da saída (classes): {output.shape}")

In [9]:
testVGG16()

Shape da entrada: torch.Size([1, 3, 224, 224])
Shape da saída (classes): torch.Size([1, 1000])


### Arquitetura AlexNet - Implemente

https://upload.wikimedia.org/wikipedia/commons/a/ad/AlexNet_block_diagram.svg

![image.svg](https://upload.wikimedia.org/wikipedia/commons/a/ad/AlexNet_block_diagram.svg)


In [ ]:
## Ex1

### GoogLeNet

https://arxiv.org/pdf/1409.4842

![image.png](https://miro.medium.com/v2/resize:fit:3968/1*G9ir2O2wEcSZtglhcWsnNA.png)

In [ ]:
##Ex2

## Extra - Manifold Hyper Connections

![image.png](https://miro.medium.com/v2/resize:fit:1400/1*hHao6kI3HF1wfFV00XKxLw.png)

In [11]:
class mHCLayer(nn.Module):
    def __init__(self, n_streams, d_model, functional_block):
        super(mHCLayer, self).__init__()
        self.n = n_streams
        self.C = d_model
        self.F = functional_block

        ## Esses valores abaixos são para calcular H^pre, H^post e H^res
        # no artigo é detalhado o cálculo usando função tangente hiperbólica

        ## Matriz de pesos
        self.theta_pre = nn.Parameter(torch.randn(self.C, 1))
        self.theta_post = nn.Parameter(torch.randn(self.C,1))
        self.theta_res = nn.Parameter(torch.randn(self.C, self.n))

        ## Matriz de bias
        self.bias_pre = nn.Parameter(torch.zeros(self.n, 1))
        self.bias_post = nn.Parameter(torch.zeros(self.n,1))
        self.bias_res = nn.Parameter(torch.zeros(self.n, self.n))

        ## Parametros aprendidos
        self.alpha_pre = nn.Parameter(torch.tensor(0.1))
        self.alpha_post = nn.Parameter(torch.tensor(0.1))
        self.alpha_res = nn.Parameter(torch.tensor(0.1))

    def sinkhorn_knopp(self, W, iterations=20):
        # Faz a transformação para o manifold
        M = torch.exp(W)
        for _ in range(iterations):
            M = M / (M.sum(dim=-1, keepdim=True) + 1e-12)
            M = M / (M.sum(dim=-2, keepdim=True) + 1e-12)
        return M

    def forward(self, x_l):
        # (RMSNorm) fator de escala para evitar saturação
        r = torch.norm(x_l, dim=-1, keepdim=True) / (self.n * self.C)**0.25
        x_norm = (1.0/r) * x_l

        ## Computando matrizes pre, post e res
        # Estamos usando sigmoid ao invés da tanh para evitar cancelamento de
        # sinal (está no artigo também) -> força ser positivo
        H_pre = torch.sigmoid(x_norm @ self.theta_pre) + self.bias_pre
        H_post = torch.sigmoid(x_norm @ self.theta_post) + self.bias_post
        H_res_unnormalized = torch.sigmoid(x_norm @ self.theta_res) + self.bias_res
        H_res = self.sinkhorn_knopp(H_res_unnormalized)

        layer_input = torch.bmm(H_pre.transpose(1, 2), x_l)
        layer_output = self.F(layer_input)
        update_part = torch.bmm(H_post, layer_output)
        res_part = torch.bmm(H_res, x_l)
        return res_part + update_part

In [16]:
def test_mhc_layer():
    torch.manual_seed(42)
    B = 4          # batch size
    n = 8          # número de streams
    C = 16         # dimensão do modelo

    # Bloco funcional simples
    class DummyBlock(nn.Module):
        def __init__(self, C):
            super().__init__()
            self.linear = nn.Linear(C, C)

        def forward(self, x):
            return self.linear(x)

    F = DummyBlock(C)
    layer = mHCLayer(n_streams=n, d_model=C, functional_block=F)
    x = torch.randn(B, n, C)

    print(f"Input shape: {x.shape}")
    y = layer(x)
    print(f"Output shape: {y.shape}")
    assert y.shape == (B, n, C), "Shape de saída incorreto"
    print("Shapes consistentes.")

In [17]:
test_mhc_layer()

Input shape: torch.Size([4, 8, 16])
Output shape: torch.Size([4, 8, 16])
Shapes consistentes.
